#### Importing pandas, numpy, seaborn, scipy, sklearn, and matplotlib. These functions are provided in Python. Using these function, people can create different types of data, and data structures.

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
from scipy import stats
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

#### The os.getcwd() function helps to get current working directory. The reason for identifying the current working directory is to make sure this jupyter notebook is in the folder you want it to be. A wrong folder may lead to unexpected results

In [ ]:
import os
os.getcwd()

#### You notice that, in the next cell, we use relative path to read the csv file. This is because, by checking the current working directory, the Murder.csv is in the same folder as this jupyter notebook. 

In [ ]:
data = pd.read_csv('Murder.csv') # Read .csv dataset

In [ ]:
data.head() # Check the first five rows of the data

#### checking the shape is important in some cases. It helps you to get the dimension information about the dataset

In [ ]:
data.shape # Check the shape

#### checking the null values is extremely important. Null values directly affect the data preprocessing and model performance 

In [ ]:
data.isnull().any() # Check the NaN value

#### train_test_split() is an useful function that helps to split the data to training dataset, and testing dataset. You should be familiar with it. 

To know more about it, check https://www.geeksforgeeks.org/how-to-do-train-test-split-using-sklearn-in-python/

In [ ]:
#Split the dataset into two parts, 90% for training and 10% for testing
#Assign an integer to random_state and it is the seed used by the random number generator 
train, test = train_test_split(data, test_size = 0.1, random_state = 0) 
print('Size of the train dataset is ', train.shape)
print('Size of the test dataset is ', test.shape)

#### In the next cell, you should pay attention to assign the features (x) and labels (y) correctly by using the output from the train_test_split()

#### You are expected to be familiar with it

In [ ]:
#Let x in the training set be the variable without Murder
#Let y in the training set be Murder
x_train = train.iloc[:, :-1]

#Let x in the testing set be the variable without Murder
#Let y in the testing set be Murder
y_train = train['Murder']
x_test = test.iloc[:, :-1]
y_test = test['Murder']

#### Sometimes, we can find some pattern by checking the pairwise scatter plot. For example, in the next figure, we found there exists some relationship between the illiteracy and the income. A lower income may indicate higher illiteracy. 

#### seaborn provides a API pariplot() to help to draw the pairwise scatter plot.

In [ ]:
#Use seaborn to plot pairwise scatter plot
sns.pairplot(train)

#### Instead of visualizing the relationship, the correlation quantifying it as shown in the next cell.

In [ ]:
def correlation_heatmap(data):
    _ , ax = plt.subplots(figsize =(14, 12)) #Only ax is used in following code, we do not assign value to the other variable
    colormap = sns.diverging_palette(220, 10, as_cmap = True)
    
    sns.heatmap(
        data.corr(), 
        cmap = colormap,
        square=True, 
        cbar_kws={'shrink':.9 }, 
        ax=ax,
        annot=True, 
        linewidths=0.1,vmax=1.0, linecolor='white',
        annot_kws={'fontsize':12 }
    )
    
    plt.title('Pearson Correlation of Features', y=1.05, size=15)
    plt.show()
    
correlation_heatmap(train) #Input train_df to the function, and check the correlation of all variables

#### We begin to fit the very first linear regression model by using the function 'LinearRegression()'
https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LinearRegression.html

In [ ]:
#Fit linear regression to training data
linear_regression = LinearRegression()
linear_regression.fit(x_train, y_train)

In [ ]:
print('The intercept estimated by Sklearn is ', linear_regression.intercept_)
print('The coefficients estimated by Sklearn are ',linear_regression.coef_)

#### Previously, we use sklearn function 'LinearRegression()' to fit the linear regression. In next two cells, we fit the model by using the formula shown in the lecture to calculate the coefficients.

In [ ]:
#Add a constant term in dataframe for linear regression
X_train = x_train.copy()
X_train['Constant'] = 1
X_train = X_train[['Constant', 'Population', 'Income', 'Illiteracy', 'Frost']]
X_train.head()

In [ ]:
#Coefficients estimation by formula
inverse_matrix = np.linalg.inv(np.dot(X_train.T, X_train))
hat_matrix = np.dot(inverse_matrix, X_train.T)
est_coeff = np.dot(hat_matrix, y_train)
print('Coefficients estimated by the formula is ', est_coeff)

#### Sometimes we want to check statistical properties about the model as shown below

In [ ]:
#Check the significance of parameters
import statsmodels.api as sm
lr = sm.OLS(y_train, x_train)
lr = lr.fit()
print(lr.summary())

#### Previously we fit the model using the x_train. In the next cell, we use the fitted model to make predictions based on the x_test. (recall we use train_test_split to split the data into training set and testing set)

In [ ]:
y_predict = linear_regression.predict(x_test) #Predict y based on test dataset

#### We should make sure that the linear regression model assumption is meet.  In next three cells, we draw the plot to check the three assumptions in linear regression

#### The next cell plots the QQ plot

In [ ]:
#Use scipy to plot QQ plot
#stats.probplot(y_train, dist="norm", plot=plt)
stats.probplot(y_train - linear_regression.predict(x_train), dist="norm", plot=plt)
plt.show()

#### The next cell plots the Residual Residual v.s Fitted plot

In [ ]:
residuals = y_train - linear_regression.predict(x_train)
plt.scatter(linear_regression.predict(x_train), residuals)
plt.xlabel('Fitted values')
plt.title('Residual vs Fitted')
plt.axhline(y = 0, color = 'grey', ls = '--')
plt.show()

#### The next cell plots the Scale-Location plot

In [ ]:
standardized_residuals = (residuals - residuals.mean()) / residuals.std()
plt.scatter(linear_regression.predict(x_train), np.sqrt(abs(standardized_residuals)))
plt.xlabel('Fitted values')
plt.title('Scale-Location')
plt.show()

#### Some times we want to compare the true values and the predicted values (provided by the fitted linear model). Comparing these two values can help us evaluate the accuracy of the fitted model.

#### The next three cells compare and visualize the true values and the predicted values

In [ ]:
#Create a pandas dataframe with two columns, one is the real value from our testing dataset, and the other on is the predicted values based on the regression model
compare = pd.DataFrame({'Real Value': y_test, 'Predict Value': y_predict})

#y_test has its own row index, thus we need to reset the index and drop the 'index' column
compare.reset_index(inplace = True)
compare = compare.drop(['index'], axis = 1)
compare.head()

In [ ]:
compare.plot()
plt.title('Real value V.S Predict Value')
plt.show()

In [ ]:
#Plot residuals in testing dataset
plt.plot(compare['Predict Value'] - compare['Real Value'])
plt.title('Residual Plot')
plt.show()

#### In addition to visualize the fitness, the $r^2$ and adjusted $r^2$ can be used to evaluate the goodness-of-fit

In [ ]:
#Define a function that calculates R^2 
def R2(x_train_, y_train_):
    linear_regression.fit(x_train_, y_train_)
    
    LRS_ = linear_regression.score(x_train_, y_train_) #Return R^2  of the model by sklearn

    print('R^2 calculated by the library is ', LRS_)

    return LRS_

In [ ]:
print('Murder = w0 + w1*Population')
r1 = R2(x_train[['Population']], y_train)
print('-'*40)
print('Murder = w0 + w1*Population + w2*Illiteracy')
r2 = R2(x_train[['Population', 'Illiteracy']], y_train)
print('-'*40)
print('Murder = w0 + w1*Population + w2*Income + w3*Illiteracy')
r3 = R2(x_train[['Population', 'Income', 'Illiteracy']], y_train)
print('-'*40)
print('Murder = w0 + w1*Population + w2*Income + w3*Illiteracy + w4*Frost')
r4 = R2(x_train, y_train)
print('-'*40)

In [ ]:
#Define a function that calculates adjusted R^2 
def AdjR2(x_train_, y_train_, r_squared):
    adjusted_r_squared_ = 1 - (1-r_squared)*(len(y_train_)-1)/(len(y_train_)-x_train_.shape[1]-1)
    print('Adjusted R^2 is ', adjusted_r_squared_)

print('Murder = w0 + w1*Population')
adjR1 = AdjR2(x_train[['Population']], y_train, r1)
print('-'*40)
print('Murder = w0 + w1*Population + w2*Illiteracy')
adjR2 = AdjR2(x_train[['Population', 'Illiteracy']], y_train, r2)
print('-'*40)
print('Murder = w0 + w1*Population + w2*Income + w3*Illiteracy')
adjR3 = AdjR2(x_train[['Population', 'Income', 'Illiteracy']], y_train, r3)
print('-'*40)
print('Murder = w0 + w1*Population + w2*Income + w3*Illiteracy + w4*Frost')
adjR4 = AdjR2(x_train, y_train, r4)
print('-'*40)